# 流程一 · PTB-XL 数据准备

## 1. 导入依赖和设置路径

In [1]:
import ast
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
import scipy
from scipy.signal import butter, sosfiltfilt
import wfdb


CLASSES = ["NORM", "AFIB", "MI", "STTC"]
PRIORITY = ["AFIB", "MI", "STTC", "NORM"]
LEADS = ["I", "II", "III", "AVR", "AVL", "AVF", "V1", "V2", "V3", "V4", "V5", "V6"]
RATE = 100
N_SAMPLES = 1000
FILTER_ORDER = 4
BAND_HZ = [0.5, 40.0]
EPSILON = 1e-6


PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3'
OUTPUT_DIR = PROJECT_DIR / "1_Output"


SPLITS = ("train", "val", "test")
EXPECTED_FILES = (
    [f"{prefix}_{split}.npy" for prefix in ("X", "y") for split in SPLITS]
    + [f"meta_{split}.csv" for split in SPLITS]
    + ["patient_map.csv", "preprocessing.json"]
)

## 2. 筛选标签并划分数据

优先级为 AFIB > MI > STTC > NORM。One-hot 列顺序为 NORM、AFIB、MI、STTC。官方第 1–8、9、10 折用于训练、验证和测试。

In [2]:
def select_records(data_dir):
    records = pd.read_csv(data_dir / "ptbxl_database.csv",
                          usecols=["ecg_id", "patient_id", "filename_lr", "strat_fold",
                                   "scp_codes", "age", "sex"])
    statements = pd.read_csv(data_dir / "scp_statements.csv", index_col=0)
    diagnostic_classes = statements.loc[statements.diagnostic == 1, "diagnostic_class"].to_dict()

    def primary_label(raw_codes):
        codes = ast.literal_eval(raw_codes)
        targets = {diagnostic_classes.get(code) for code in codes}
        # Rhythm codes such as AFIB may have a likelihood of zero (unknown).
        if "AFIB" in codes:
            targets.add("AFIB")
        return next((label for label in PRIORITY if label in targets), None)

    if records.ecg_id.isna().any() or not records.ecg_id.is_unique:
        raise ValueError("Missing or duplicate ecg_id")
    if records.patient_id.isna().any() or not records.strat_fold.isin(range(1, 11)).all():
        raise ValueError("Missing patient_id or invalid strat_fold")
    if (records.groupby("patient_id").strat_fold.nunique() > 1).any():
        raise ValueError("A patient belongs to more than one official fold")
    total = len(records)
    records["label"] = records.scp_codes.map(primary_label)
    records = records.dropna(subset=["label"]).sort_values("ecg_id").copy()
    records["patient_id"] = records.patient_id.astype("int64")
    records["split"] = records.strat_fold.map({**dict.fromkeys(range(1, 9), "train"),
                                               9: "val", 10: "test"})
    if not records.sex.isin([0, 1]).all():
        raise ValueError("Unexpected sex code")
    return records, total


records, total = select_records(DATA_DIR)

## 3. 计算年龄标尺

把脱敏占位年龄 300 记作 90。缺失年龄用训练集中位数填补，三组共用训练集拟合的均值和标准差。

In [3]:
def normalize_age(records):
    # PTB-XL anonymizes the oldest age group using 300. Represent it by 90.
    ages = records.age.replace(300, 90)
    train = records.split == "train"
    median = float(ages[train].median())
    ages = ages.fillna(median)
    mean = float(ages[train].mean())
    std = float(ages[train].std(ddof=0))
    if not np.isfinite([median, mean, std]).all() or std <= 0:
        raise ValueError("Cannot fit age normalization on the training set")
    records["age_norm"] = (ages - mean) / std
    return {"sentinel": 300, "replacement": 90, "median": median, "mean": mean,
            "std": std, "ddof": 0, "missing": "fill with training median"}

age_stats = normalize_age(records)

## 4. 处理 100 Hz 波形

对每条 12 导联记录做 0.5–40 Hz 巴特沃斯双向带通滤波与逐导联 Z-score。滤波器设计参数为 N=4。

In [4]:
def read_offline_waveform(path, sos):
    signal, header = wfdb.rdsamp(str(path))
    if header["fs"] != RATE or signal.shape != (N_SAMPLES, len(LEADS)):
        raise ValueError(f"Unexpected sample rate or waveform shape at {path}")
    if header["sig_name"] != LEADS or any(unit != "mV" for unit in header["units"]):
        raise ValueError(f"Unexpected lead order or units at {path}")
    # Whole-record, zero-phase filtering is for offline analysis of complete ECGs.
    filtered = sosfiltfilt(sos, signal.T, axis=-1)
    centered = filtered - filtered.mean(axis=-1, keepdims=True)
    scaled = centered / (filtered.std(axis=-1, keepdims=True) + EPSILON)
    if not np.isfinite(scaled).all():
        raise ValueError(f"Non-finite signal at {path}")
    return scaled.astype(np.float32)

## 5. 导出训练、验证和测试文件

每组生成一份波形、一份标签和一份元数据。参数文件同时记录类别先验、导联顺序和处理配置。

In [5]:
def write_split(data_dir, output_dir, split, frame, sos):
    if frame.empty:
        raise ValueError(f"No records in split {split}")
    indices = pd.Categorical(frame.label, categories=CLASSES).codes
    labels = np.eye(len(CLASSES), dtype=np.int64)[indices]
    np.save(output_dir / f"y_{split}.npy", labels)
    frame[["ecg_id", "age_norm", "sex"]].to_csv(output_dir / f"meta_{split}.csv", index=False)

    waveforms = np.lib.format.open_memmap(
        output_dir / f"X_{split}.npy", mode="w+", dtype=np.float32,
        shape=(len(frame), len(LEADS), N_SAMPLES),
    )
    for index, record in enumerate(frame.itertuples(index=False)):
        waveforms[index] = read_offline_waveform(data_dir / record.filename_lr, sos)
        if (index + 1) % 3000 == 0:
            print(f"{split}: {index + 1}/{len(frame)}", flush=True)
    waveforms.flush()
    counts = labels.sum(axis=0).tolist()
    print(f"{split}: {len(frame)} ECGs, counts {counts}", flush=True)
    return {"count": len(frame), "patients": int(frame.patient_id.nunique()),
            "class_counts": dict(zip(CLASSES, counts))}


def prepare(data_dir, output_dir, records, total, age_stats):
    if output_dir.exists():
        print(f"Output directory already exists. Skipping export: {output_dir}")
        return

    sos = butter(FILTER_ORDER, BAND_HZ, btype="bandpass", fs=RATE, output="sos")
    output_dir.parent.mkdir(parents=True, exist_ok=True)
    # Publish only after all splits have completed successfully.
    with TemporaryDirectory(prefix=".ptbxl-", dir=output_dir.parent) as temp:
        staged = Path(temp) / "output"
        staged.mkdir()
        splits = {}
        for split in SPLITS:
            frame = records.loc[records.split == split]
            splits[split] = write_split(data_dir, staged, split, frame, sos)
        records[["ecg_id", "patient_id"]].to_csv(staged / "patient_map.csv", index=False)
        train_labels = np.load(staged / "y_train.npy", allow_pickle=False)
        stats = {
            "source_records": total, "selected_records": len(records),
            "class_order": CLASSES, "label_priority": PRIORITY, "lead_order": LEADS,
            "splits": splits, "age": age_stats,
            "train_class_prior": train_labels.mean(axis=0).astype(np.float32).tolist(),
            "signal": {"input_unit": "mV", "output_unit": "z-score", "fs_hz": RATE,
                       "samples": N_SAMPLES, "bandpass_hz": BAND_HZ,
                       "butter_N": FILTER_ORDER, "filter": "scipy.signal.sosfiltfilt",
                       "normalization": "per record, per lead, ddof=0",
                       "std_epsilon": EPSILON, "mode": "offline; uses full 10-second record"},
            "versions": {"numpy": np.__version__, "pandas": pd.__version__,
                         "scipy": scipy.__version__, "wfdb": wfdb.__version__},
        }
        (staged / "preprocessing.json").write_text(json.dumps(stats, indent=2) + "\n",
                                                   encoding="utf-8")
        staged.replace(output_dir)
    print(f"Wrote {output_dir}", flush=True)


prepare(DATA_DIR, OUTPUT_DIR, records, total, age_stats)

Output directory already exists. Skipping export: /Users/forever/Desktop/AI_Project/1_Output


## 6. 核对文件和类别计数

In [6]:
missing_files = [name for name in EXPECTED_FILES if not (OUTPUT_DIR / name).is_file()]
if missing_files:
    print(f"Verification skipped. Missing output files: {', '.join(missing_files)}", flush=True)
else:
    stats = json.loads((OUTPUT_DIR / "preprocessing.json").read_text(encoding="utf-8"))
    rows = []
    for split in SPLITS:
        waveforms = np.load(OUTPUT_DIR / f"X_{split}.npy", mmap_mode="r", allow_pickle=False)
        labels = np.load(OUTPUT_DIR / f"y_{split}.npy", allow_pickle=False)
        metadata = pd.read_csv(OUTPUT_DIR / f"meta_{split}.csv")
        selected = records.loc[records.split == split]
        expected_labels = np.eye(len(CLASSES), dtype=np.int64)[
            pd.Categorical(selected.label, categories=CLASSES).codes
        ]
        if waveforms.shape != (len(selected), len(LEADS), N_SAMPLES) or waveforms.dtype != np.float32:
            raise ValueError(f"{split} unexpected waveform shape or dtype")
        np.testing.assert_array_equal(labels, expected_labels)
        np.testing.assert_array_equal(metadata.ecg_id.to_numpy(), selected.ecg_id.to_numpy())
        rows.append({"split": split, "Records": len(metadata), "X.shape": waveforms.shape,
                     **dict(zip(CLASSES, labels.sum(axis=0).tolist()))})

    print("Training class priors", stats["train_class_prior"])
    print(pd.DataFrame(rows).to_string(index=False))

Training class priors [0.49426567554473877, 0.07936299592256546, 0.2578805983066559, 0.1684907227754593]
split  Records           X.shape  NORM  AFIB   MI  STTC
train    15259 (15259, 12, 1000)  7542  1211 3935  2571
  val     1891  (1891, 12, 1000)   946   151  461   333
 test     1914  (1914, 12, 1000)   956   152  468   338
